# Aprendizaje No Supervisado — Resumen visual
### Predicción del Éxito Académico en Educación Superior

**Técnicas:** PCA (reducción de dimensionalidad) + K-Means (clustering)  
**Variables de entrada:** 14 variables numéricas seleccionadas (rendimiento académico, perfil de acceso, situación económica)  
**Estrategia de reducción:** PCA con umbral de varianza acumulada del 80% → N componentes retenidos  
**Clustering:** K-Means sobre el espacio PCA reducido — visualización en PC1 vs PC2  
**Elección de K:** método del codo + Silhouette Score  
**Estabilidad:** ARI medio sobre 30 ejecuciones con semillas distintas  
**Objetivo:** descubrir estructura latente en los datos sin usar las etiquetas reales

---
## 1. PCA — Scree Plot

![01_scree_plot.png](../figures/01_scree_plot.png)

**¿Qué muestra?**
Antes de explicar hay que recordar que con PCA resolvíamos el problema de autovalores y autovectores de la matriz de inputs (que hemos reducido a 14 features). Los valores de los autovalores medían cuánta varianza captaba esa dirección en el espacio y los autovectores eran esas direcciones concretas en el espacio (lo que llamamos loadings). Lo que queremos es saber cuántas direcciones queremos mantener en nuestro nuevo espacio sobre el que vamos a proyectar todos nuestros datos. 
Por ello, analizamos cuánta varianza acumulan los componentes principales que son vectores con las proyecciones de los datos sobre cada loading (direcciones).

Entonces aquí vemos cuánta varianza hay explicada en cada uno de esos componentes principales y en la gráfica de la derecha la varianza acumulada a medida que vamos tomando ese componentes principales

**¿Cuántos componentes se retienen y por qué?**
Nos quedamos con 6 componentes ya que al tomarlos ya tenemos más de 80% de la varianza total explicada que es un umbral coherente a la hora de buscar un espacio en el que reducimos dimensionalidad (6 < 14) y queremos mantener bastante varianza del total del dataset.

Esto significa que vamos a proyectar nuestro dataset sobre un espacio de 6 dimensiones generado por las 6 direcciones del espacio que más varianzan captan. Las columnas de este nuevo dataset son las componentes principales que son vectores cuyos elementos son la proyeccion de cada dato sobre el loading. 

Por ejemplo, el primer elemento de la primera columna será la proyección de la primera fila del dataset sobre la dirección de máxima varianza (phi 1).

**Varianza explicada por PC1 y PC2 — ¿qué implica para la visualización 2D?**
Entre las dos primeras componentes sumamos 53.2% de la varianza total. Eso significa que en la proyección 2D que usaremos para visualizar los clusters tendremos más de la mitad de la información total del subconjunto de 14 variables. Por tanto es una proyección bastante informativa.
Sin embargo, hay que tener en cuenta que la varianza capturada por los componentes 3 a 6 no es visible en el plano 2D.
El clustering se aplica sobre los 6 componentes completos y la visualización en PC1 y PC2 es solo una proyección por lo que veremos en los scatter plots siguientes es representativo de la estructura principal pero no recoge toda la información que hemos usado en K-Means

---
## 2. PCA — Proyección 2D coloreada por etiqueta real

![02_pca_scatter_target.png](../figures/02_pca_scatter_target.png)

**¿Qué muestra la proyección sobre PC1 y PC2?**
Veremos en la siguiente gráfica con más profundidad que direcciones representan PC1 y PC2. De momento nos quedamos con que PC1 es el eje de rendimiento académico y PC2 separa otro eje de variación que analizaremos mejor en la siguiente gráfica.

Aún no hemos aplicado K-Means es simplemente un paso previo al clustering para ver si en el espacio no supervisado de 6 dimensiones (solo 2 proyectadas aquí) ya aparece alguna separación entre las clases reales. 

**Separabilidad por clase — ¿qué clases se separan bien y cuáles se solapan?**
Vemos que los graduados se concentran claramente con PC1 positivo y PC2 negativo. Son estudiantes con alto rendimiento académico y tienen ese valor negativo en PC2 que entenderemos mejor cuándo analizemos más en profundidad sobre qué dirección se mueve el loading de ese PC2.

Los abandonos dominan con PC1 negativo y PC2 positivo.

Los matriculados están dispersos por el centro, solapando con ambas clases. Esto no nos sorprende nada ya que en clasificación ya detectamos que la clase matriculados tien una naturaleza semántica complicada al ser un perfil académico en transito muy solapado con perfiles de graduados y abandono.

**¿Qué nos adelanta esta visualización sobre la dificultad del clustering?**
Lo bueno es que sí vemos que hay alguna estructura real, las clases no están distribuidas aleatoriamente en el espacio PCA obtenido y eso es bueno. Sin embargo, K-Means no tiene por objetivo predecir por las etiquetas Y de los modelos supervisados. K-Means encontrará ciertas agrupaciones que podrán coincidir o no exactamente con las Y supervisadas.

---
## 3. PCA — Loadings de PC1 y PC2

![03_pca_loadings.png](../figures/03_pca_loadings.png)

**¿Qué mide un loading y cómo se interpreta su signo y magnitud?**
Cada componente principal es un vector en el espacio original de 14 dimensiones. El peso de cada posición dentro de cada vector loading es el peso que tiene la feature original dentro de ese vector dirección.

Por ejemplo el loading, la posición j del loading phi 1 nos dice cuánto tira la variable j del dataset en esa dirección de máxima varianza.

Cuánto mayor sea el valor del elemento j del loading más importante será la feature j para definir esa dirección. Si es positivo significa que valores altos de esa variable empujan a valores alto del componente y con negativo al revés.


**¿Qué variables dominan PC1? ¿Qué dimensión del estudiante captura?**
Vemos algo clarísimo, PC1 está dominadísimo por el bloque de rendimiento académico universitario. Muchas features académicas positivas con pesos similares. 
Esto significa que un estudiante con PC1 alto ha cursado muchas asignatures, las ha aprobado y tiene buenas notas en ambos semestres. Un alumno con PC1 muy negativo básicamente no tiene casi actividad universitaria registrada. Pueden luego ser graduados o matriculados ya que a menudo hay poca actividad académica en universitarios que vienen de otras carreas y ya tienen muchas asignaturas convalidadas y que se graduan pero que académicamente tienen perfiles en PC1 muy pobres.

Concluímos que la dirección de mayor varianza en este dataset no se explica por el perfil socioeconómico sino exclusivamente por lo que el estudiante hace dentro de la universidad

**¿Qué variables dominan PC2? ¿Qué dimensión complementaria aporta?**
PC2 mezcla bloques más distintos. PC2 opone dos perfiles de estudiantes. En el extremo positivo está el estudiante mayor que no tiene pagos al día y que se presenta a muchas asignaturas aunque sus notas previas no eran buenas. En el extremo negativo tenemos estudiantes jóvenes con un buen expediente previo, becados y con matricula al día.

Lo interesante realmente es que esta dirección capta una dimensión socioeconómica y de perfil de acceso que es ortogonal al rendimiento académico puro que hemos encontrado en PC1. Estas direcciones claras separadas por features que sí tienen interes verlos por separado también nace de haber hecho una buena selección de variables antes de aplicar PCA.



**Coherencia con el análisis de colinealidad de modelos supervisados:**
Todo esto es extremadamente coherente con lo que habíamos visto en clasificación y regresión. Usábamos siempre Lasso que buscaba eliminar esa colinealidad que vemos muy claramente entre features académicas con loadings casi idénticos en PC1.
PCA nos ayuda justamente también a combatir ese colinealidad problemática fusionando en un único eje.

Vemos claramente cómo el aprendizaje no supervisado también valida las conclusiones de los modelos supervisados. De hecho muchas veces es un paso clave antes de siquiera pasar a modelos supervisados porque permite entender muy bien la geometría del problema y reducir multicolinealidades (que en caso de no regularizar con Lasso por ejemplo) sería problemáticas.


---
## 4. K-Means — Elección de K óptimo (Codo + Silhouette)

![04_elbow_silhouette.png](../figures/04_elbow_silhouette.png)

En las gráficas anteriores hemos terminado de analizar el nuevo espacio de representación devuelto por PCA. Ahora, aplicamos K-Means sobre los 6 componentes retenidos para encontrar agrupaciones naturales. 
La principal debilidad de K-Means es que necesita que le digamos cuántos clusters buscar. Para ello usamos estos criterios complementarios

**¿Qué mide el método del codo y qué nos dice su curva?**
Vimos que con el método del codo podíamos medir diametro del cluster o WSS. Aquí vamos a trabajar con la inercia que es la suma de las distancia al cuadrado de cada punto a su centroide más cercano, es decir WSS. Esto mide qué tan compactos son los clusters internamente. Cuanto menor es la inercia, más juntos están los puntos dentro de cada cluster.

La cosa es que la inercia siempre decrece al aumentar K pero no buscamos el mínimo sino el codo: es el punto donde la curva se aplana bruscamente indicando que añadir más clusters ya no compensa.

Vemos que el codo más claro es en K=3
Sin embargo, esta medida solo tiene en cuenta la compactitud dentro de los clusters. Sin embargo, Silhouette tiene en cuenta compactitud + separación máxima entre clusters. Por tanto usaremos Silhouette para tomar la decisió final

**¿Qué mide el Silhouette Score y cómo complementa al codo?**
Al tener en cuenta distancia entre clusters y compactitud tenemos una medida más rica. Vemos que de todos modos, el valor óptimo es 0.39 que es relativamente moderado (1 es perfecto). Eso tiene sentido ya que en la representación en 2D ya intuíamos que los clusters no iban a poder estar tan separados.

**K óptimo elegido y justificación:**
El valor elegido es K=3. Ambos criterios concuerdan en que 3 clusters es suficiente y óptimo.

**¿Por qué el K elegido coincide o no con el número de clases reales (3)?**
Vemos que hay tantos clusters como clases reales y eso no es una coincidencia. K-Means no ha visto nunca las etiquetas reales y ha trabajado exclusivamente con la geometría del espacio en 6 dimensiones de PCA. El hecho de que encuentre 3 grupos sugiere que las tres situaciones académicas que predecíamos tienen una estructura geométrica real en el espacio de features.

Es verdad que los clusters no van a recuperar exactamente las tres clases porque ya sabíamos que el solapamiento de Matriculado en el espacio es problemático pero ya es interesante ver que sí se reconocen 3 grupos en el espacio de features.

Lo que queda claro es que los perfiles de rendimiento y la situación económica de los estudiantes se agrupan naturalmente en tres regiones diferenciadas. 

---
## 5. K-Means — Clusters vs Etiquetas reales (side by side)

![05_clusters_vs_target.png](../figures/05_clusters_vs_target.png)

**¿Qué muestra la comparativa lado a lado?**
Aquí llegamos al punto más interesante. En el lado izquierdo tenemos los 3 clusters hallados por K-Means trabajando solo con la geometría del espacio de PCA. 
En el panel derecho tenemos las etiquetas reales supervisadas. Así vemos dónde coinciden más o menos.

**¿Qué clusters coinciden con qué etiquetas reales? ¿Hay correspondencia clara?**
El cluster azul ocupa la zona con PC1 negativo que es exactamente la zona en dónde se concentran los abandonos. Es la correspondencia más limpia de las tres. Eso también es algo que habíamos visto al hacer el análisis SMOTE en supervisado. Vimos que las métricas sobre abandono no se veían tan afectadas al usar SMOTE y eso inducía a pensar que es el grupo más claramente separable del dataset y lo volvemos a encontrar aquí. Personas con actividad académica muy poco activa y situación económica media/baja.

El cluster verde es el más grande y ocupa la zona con PC1 medio-alto y PC2 bajo que es donde se concentraban los graduados. Sin embargo absorbe muchos matriculados del centro. Este cluster representa estudiantes con actividad académica bastante alta y perfil socioeconómico cómodo. Es normal que convivan matriculados y abandonos en este cluster porque son perfiles de estudiantes muy parecidos, independientemente de lo que terminen haciendo.

Por último, el cluster amarillo es más pequeño pero muy interesante. Son estudiantes con mucha actividad académica pero también con edad alta y posibles deudas. No corresponde limpiamente a ninguna de las tres clases supervisadas. Esto sugiere que K-Means ha encontrado un subgrupo que con la etiquetación supervisada no distinguíamos y que es muy interesante.

**¿Dónde se mezclan clusters y etiquetas? ¿Qué nos dice sobre la estructura del problema?**
Los Matriculados son la clase que más se mezcla en ambos paneles por la misma razón que ya detectamos en clasificación: son observaciones en tránsito con perfiles académicos aún no diferenciados completamente de graduados ni abandonos.

**Limitación de visualizar en 2D un clustering hecho en N dimensiones:**
Es importante aclarar que K-Means ha operado en el espacio de 6 componentes PCA. Sin embargo la representación es en 2D que representan parcialmente los resultados. Sin embargo lo hacemos así ya que las fronteras reales entre clusters son en 6 dimensiones y no podemos visualizarlas correctamente.

Por eso es clave analizar el heatmap de los perfiles en la siguiente gráfica ya que permite interpretar los clusters en el espacio original de variables sin esa distorsión de la proyección

---
## 6. K-Means — Perfiles de clusters (Heatmap)

![06_cluster_profiles_heatmap.png](../figures/06_cluster_profiles_heatmap.png)

**¿Qué representa cada celda del heatmap?**
Cada celda es la desviación de la media de esa variable en ese cluster respecto a la media global del dataset.

Un valor de +2.8 indica que ese cluster tiene de media 2.8 desviaciones típicas por encima de la media global en esa variable. El color verde indica por encima de la media y el color rojo por debajo.

Esto nos permite leer directamente el perfil de cada cluster sin necesidad de recordar las escalas originales de cada variable

**¿Qué perfil académico/socioeconómico define a cada cluster?**
Cluster 0 - (Amarillo) - Estudiante con alto rendimiento y mayor edad:
Es el cluster más pequeño y el más extremo en rendimiento. Presenta desviaciones muy altas en todo el bloque académico. Sus notas medias también están muy por encima. En el perfil socioeconómico destaca que estos estudiantes son significativamente mayores que la media. Con la matriz vemos que la mayoría son graduados. Este grupo es el que la etiquetación supervisada no distinguía.

Cluster 1 - (Azul)- Estudiante en riesgo de abandono
Es el cluster más claramente definido y el más alejado de la media en sentido negativo. La actividad académica es muy negativa sin excepción. Además, en el eje socio-económico tienen también un perfil socioeconómico también ligeramente peor que los otros perfiles de estudiantes. El cluster 1 es el grupo más nitidamente definido con casi 80% de abandonos con bajo rendimiento académico generalizado y dificultados económicas.

Cluster 2 (Verde) - Estudiante típico
Es el cluster mayoritario y el más cercano de la media global en casi todas las variables. No es un perfil extremo en ninguna dirección y la matriz nos deja entender que es la distribución más heterogénea dentro de los clusters. Los abandonos y matriculados se solapan mucho y caen mucho en un mismo cluster ya que tienen esos perfiles geométricos muy cercanos en el espacio de features. Por eso en KNN sobre todo obteníamos métricas tan complicadas entre estas clases.

**Coherencia con la feature importance de Random Forest**
En Random Forest asignaturas_2sem_aprobadas lideraba el feature importance y aquí también aparece como una de las variables más discriminativa entre clusters. 

---
## 7. Estabilidad del clustering — ARI sobre 30 ejecuciones

![07_stability_ari.png](../figures/07_stability_ari.png)

K-Means tiene un problema y es que su resultado depende de cómo se inicializan los centroides al princpio. Con semillas distintas puede converger a soluciones distintas. Lo que queremos ver es si los clusters que hemos encontrados son robustos o son un accidente de la inicialización que hemos usado

Para verlo, repetimos K-Means 30 veces cada vez con semilla aleatoria distinta y comparamos resultado sntre sí. El ARI es la métrica que cuantifica esta comparación

**¿Qué mide el ARI (Adjusted Rand Index) y por qué es una buena métrica de estabilidad?**
ARI compara dos asignaciones de clusters y mide cuánto se parece. Si tenemos por ejemplo dos ejecuciones de K-Means, ARI pregunta si los pares de estudiantes que estaban juntos en la primera ejecución lo están también en la segunda.

Un ARI 0 significa que la particiones no se parece nada y una ARI 1 significa que las ejecuciones produjeron exactamente la misma partición.

**¿Cómo se interpreta la distribución observada (media, dispersión)?**
Vemos que la distribución tiene dos grupos muy claramente distintos. Un grupo de pares tiene ARI = 0.6 y otro ARI = 1.

Esto significa que K-Means no siempre converge a la misma solución. A veces encuentra la solución A y otras veces la solución B. Cuando comparas A con A o B con B el ARI es 1 pero al comparar A ocn B el ARI es 0.6.

Esto nos dice que hay dos soluciones que existen y son algo distintas.

**¿Qué nos dice la estabilidad sobre la estructura real de los datos?**
Esto nos dice algo muy interesante: el dataset tiene dos particiones con K=3 que son bastante buenas en términos de inercia. K-Means elige una entre ellas según la inicialización. Esto ocurre bastante cuando hay una zona del espacio donde la frontera entre clusters es ambiguo que es el claso entre el cluster 0 y 2 que vemos visualmente.

Dependiendo de dónde caigan los centroides iniciales algunos estudiantes de esa frontera entra en un cluster u otro.


---
## 8. Conclusión — Aprendizaje No Supervisado

<div style="border: 2px solid #2e86c1; border-radius: 8px; padding: 20px 24px; margin: 28px 0; background-color: #eaf4fb; color: #000000;">
<h3 style="color: #1a5276; margin-top: 0; padding-bottom: 8px; border-bottom: 1px solid #aed6f1; margin-bottom: 14px;">Conclusiones — Aprendizaje No Supervisado</h3>

El análisis no supervisado ha aportado una perspectiva complementaria e independiente de los modelos supervisados.

PCA nos ha permitido reducir el espacio original de 14 variables a 6 componentes que retienen 83.6% de la varianza total. Es una forma de limpiar el espacio de features y eliminar dimensiones de menor varianza que suelen captar ruido y así poder trabajar sobre dimensiones realmente informativas. Las dos primeras componentes acumulan 53.2% de la varianza total, lo que hace que la proyección 2D sea suficientemente representativa para visualizar la estructura de los datos. El hecho de poder ver agrupaciones en ese plano 2D antes de aplicar K-Means ya es muy buena señal de que el espacio de features tiene una estructura geométrica real

Al analizar los loadings hemos visto que las dos direcciones de máxima varianza capturan dimensiones que son semánticamente ortogonales: PC1 recoge el rendimiento académico universitario y PC2 captura una dimensión socioeconómica. Esta ortogonalidad entre los dos primers componentes es especialmente valiosa porque resuelva parcialmente la multicolinealidad elevado que sabemos que tiene nuestro dataset por lo que vimos en los modelos supervisados. Esto es una de las razones por las que PCA se aplica habitualmente como paso previo al clustering.

La elección de K=3 con el método del codo y el Silhouette Score no es un resultado trivial. K-Means ha encontrado espontáneamente sin ver las etiquetas que el espacio de 6 dimensiones tiene tres agrupaciones naturales que coincide exactamente con el número de clases que predecíamos  en el problema supervisado. Esto confirma que las situaciones académicas Abandono, Matriculado y Graduado tienen alguna estructura geométrica real en el espacio de features. Es importante matizar que los clusters no recuperan exactamente esas clases, de hecho el solapamiento de Matriculados que ya intuimos en los modelos supervisados se manifiesta aquí como la principal fuente de ambiguedad geométrica.

La comparativa entre clusters y etiquetas reales nos ha permitido identificar tres perfiles de estudiantes. El cluster 1 es un perfil de riesgo de abandono con bajo rendimiento académico y condiciones socioeconómicas desfavorables. Casi el 80% de sus miembros son Abandonos. El cluster 2 es el más grande y representa el alumno estándar y una composición heterogénea de Graduados y Matriculados que refleja que en este grupo no hay suficiente separación geométrica para distinguir quién acabará graduándose. El cluster 0 es el más interesante, es algo completamente nuevo: un grupo pequeño de estudiantes con rendimiento académico muy por encima de la media y con edad también bastante por encima. Es un subgrupo que la etiquetación  supervisada no distinguía explícitamente y que K-Means ha identificado.

Finalmente, el análsis de estabilidad con ARI sobre 30 inicializaciones aleatorias no ha revelado que K-Means converge a dos soluciones distintas según la inicialización y ambas buenas en términos de inercia. La inestabilidad afecta a la frontera difusa entre los cluster 0 y 2 donde el espacio de features no tiene separación geométrica nítida.

</div>